<div dir="rtl" style="text-align: right;">

# תכנות אסינכרוני עם asyncio

</div>

![Python Course](images/logo.jpg)

<div dir="rtl" style="text-align: right;">

---

## מה נלמד היום?

- מה זה תכנות אסינכרוני ולמה זה שימושי
- ההבדל בין Threading ל-asyncio
- פונקציות async ו-await
- הרצה של מספר משימות במקביל
- דוגמאות מעשיות: הורדת קבצים, קריאה ל-APIs

---

## מה זה תכנות אסינכרוני?

**תכנות אסינכרוני** מאפשר לתוכנה לבצע מספר פעולות במקביל **מבלי** לחכות שכל פעולה תסתיים.

### דוגמה מהחיים:

**תכנות רגיל (סינכרוני):**
1. שולח בקשה לשרת → מחכה לתשובה
2. שולח בקשה שנייה → מחכה לתשובה
3. שולח בקשה שלישית → מחכה לתשובה

⏱️ זמן כולל: 3 שניות (1 שנייה לכל בקשה)

**תכנות אסינכרוני:**
1. שולח שלוש בקשות **ביחד**
2. כולן מתבצעות במקביל
3. מקבל את כל התשובות

⏱️ זמן כולל: 1 שנייה!

---

## Threading vs asyncio - מה ההבדל?

| **Threading** | **asyncio** |
|--------------|-------------|
| משתמש ב-threads (חוטי ביצוע) | משתמש ב-coroutines (פונקציות מיוחדות) |
| מתאים למשימות כבדות (חישובים) | מתאים למשימות I/O (רשת, קבצים) |
| יותר משאבים (זיכרון) | פחות משאבים |
| קשה יותר לניהול | קל יותר לניהול |

**מתי להשתמש ב-asyncio?**
- הורדת קבצים מהאינטרנט
- שליחת בקשות ל-APIs
- קריאה/כתיבה של קבצים גדולים
- כל דבר שכולל המתנה (רשת, דיסק)

</div>

<div dir="rtl" style="text-align: right;">

## הבסיס: פונקציות async ו-await

### פונקציה רגילה vs פונקציה אסינכרונית

**פונקציה רגילה:**
```python
def say_hello():
    return "שלום!"
```

**פונקציה אסינכרונית:**
```python
async def say_hello():
    return "שלום!"
```

ההבדל: המילה `async` לפני `def`.

### await - המתן לפעולה אסינכרונית

`await` אומר לפייתון: "תמתין כאן עד שהפעולה הזאת תסתיים, אבל תוך כדי אפשר לעשות דברים אחרים".

</div>

In [ ]:
import asyncio
import time

# פונקציה אסינכרונית פשוטה
async def say_after(delay, message):
    """מדפיסה הודעה אחרי המתנה מסוימת"""
    await asyncio.sleep(delay)  # מחכה delay שניות (אסינכרוני!)
    print(message)
    return message

<div dir="rtl" style="text-align: right;">

**הסבר על הקוד:**
- `async def` - מגדירים פונקציה אסינכרונית
- `await asyncio.sleep(delay)` - ממתינים בצורה אסינכרונית (לא חוסמים את התוכנה!)
- שימו לב: `asyncio.sleep()` זה **לא** `time.sleep()` הרגיל

### איך מריצים פונקציה אסינכרונית?

לא אפשר פשוט לקרוא לה! צריך להשתמש ב-`asyncio.run()`:

</div>

In [ ]:
# דרך 1: הרצה פשוטה
async def main():
    print("מתחילים!")
    result = await say_after(1, "שלום אחרי שנייה")
    print(f"התוצאה: {result}")

# הרצת הפונקציה
asyncio.run(main())

<div dir="rtl" style="text-align: right;">

**מה קרה כאן?**
1. הדפסנו "מתחילים!"
2. המתנו שנייה אחת (אסינכרונית)
3. הדפסנו "שלום אחרי שנייה"
4. קיבלנו את התוצאה

זה נראה כמו קוד רגיל... אז מה המיוחד? בואו נראה את הכוח האמיתי!

</div>

<div dir="rtl" style="text-align: right;">

## הרצת מספר משימות במקביל

### דוגמה 1: הרצה רגילה (סינכרונית)

בואו נריץ שלוש פעולות **אחת אחרי השנייה**:

</div>

In [ ]:
async def run_sequentially():
    """מריץ משימות אחת אחרי השנייה"""
    print("מתחילים הרצה רגילה...")
    start = time.time()
    
    await say_after(2, "הודעה 1 אחרי 2 שניות")
    await say_after(2, "הודעה 2 אחרי 2 שניות")
    await say_after(2, "הודעה 3 אחרי 2 שניות")
    
    elapsed = time.time() - start
    print(f"\nזמן כולל: {elapsed:.1f} שניות")

asyncio.run(run_sequentially())

<div dir="rtl" style="text-align: right;">

**תוצאה:** כ-6 שניות (2+2+2)

כל משימה ממתינה שהקודמת תסתיים.

---

### דוגמה 2: הרצה במקביל (אסינכרונית אמיתית!)

עכשיו בואו נריץ את **כולן ביחד** עם `asyncio.gather()`:

</div>

In [ ]:
async def run_concurrently():
    """מריץ משימות במקביל"""
    print("מתחילים הרצה במקביל...")
    start = time.time()
    
    # gather מריץ את כולן ביחד!
    results = await asyncio.gather(
        say_after(2, "הודעה 1 אחרי 2 שניות"),
        say_after(2, "הודעה 2 אחרי 2 שניות"),
        say_after(2, "הודעה 3 אחרי 2 שניות")
    )
    
    elapsed = time.time() - start
    print(f"\nזמן כולל: {elapsed:.1f} שניות")
    print(f"תוצאות: {results}")

asyncio.run(run_concurrently())

<div dir="rtl" style="text-align: right;">

**תוצאה:** כ-2 שניות!

**מה קרה?**
- `asyncio.gather()` הריץ את שלושת הפונקציות **ביחד**
- כולן התחילו באותו זמן
- המתנו רק 2 שניות (במקום 6!)
- קיבלנו רשימה עם כל התוצאות

**זה הכוח של asyncio!** 🚀

</div>

<div dir="rtl" style="text-align: right;">

## דוגמה מעשית: הורדת מספר קבצים

בואו נראה דוגמה אמיתית - הורדת תמונות מהאינטרנט:

</div>

In [ ]:
import aiohttp  # ספרייה אסינכרונית לבקשות HTTP
import asyncio

async def download_image(session, url, filename):
    """מוריד תמונה מ-URL ושומר לקובץ"""
    print(f"מתחיל הורדה: {filename}")
    
    async with session.get(url) as response:
        content = await response.read()
        
        # שומר לקובץ
        with open(filename, 'wb') as f:
            f.write(content)
        
        print(f"הורדה הושלמה: {filename}")
        return filename

<div dir="rtl" style="text-align: right;">

**הסבר על הקוד:**
- `aiohttp` - ספרייה לבקשות HTTP אסינכרוניות (כמו requests אבל אסינכרוני)
- `async with` - context manager אסינכרוני
- `await response.read()` - קורא את התוכן בצורה אסינכרונית

עכשיו בואו נוריד מספר תמונות **במקביל**:

</div>

In [ ]:
async def download_multiple_images():
    """מוריד מספר תמונות במקביל"""
    
    # רשימת URLs לדוגמה (תמונות אקראיות)
    images = [
        ("https://picsum.photos/200/300?random=1", "image1.jpg"),
        ("https://picsum.photos/200/300?random=2", "image2.jpg"),
        ("https://picsum.photos/200/300?random=3", "image3.jpg"),
        ("https://picsum.photos/200/300?random=4", "image4.jpg"),
        ("https://picsum.photos/200/300?random=5", "image5.jpg")
    ]
    
    start = time.time()
    
    # יוצרים session אחד לכל ההורדות
    async with aiohttp.ClientSession() as session:
        # יוצרים רשימה של משימות
        tasks = [
            download_image(session, url, filename)
            for url, filename in images
        ]
        
        # מריצים את כולן במקביל!
        results = await asyncio.gather(*tasks)
    
    elapsed = time.time() - start
    print(f"\nהורדת {len(results)} תמונות במקביל לקחה: {elapsed:.1f} שניות")

# הרצה
# asyncio.run(download_multiple_images())

<div dir="rtl" style="text-align: right;">

**שימו לב:**
- הקוד מוגש אבל לא רץ (בגלל `#` לפני `asyncio.run`)
- כדי להריץ, הסירו את ה-`#` והריצו
- **התקנה:** אם אין לכם `aiohttp`, הריצו: `pip install aiohttp`

**למה זה מהיר?**
במקום להוריד תמונה אחת, לחכות, להוריד עוד אחת... כולן מורדות **ביחד**!

</div>

<div dir="rtl" style="text-align: right;">

## דוגמה נוספת: שאילתות ל-APIs במקביל

בואו נשלח מספר בקשות ל-API במקביל:

</div>

In [ ]:
async def fetch_user(session, user_id):
    """מביא מידע על משתמש מ-API"""
    url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
    
    async with session.get(url) as response:
        data = await response.json()
        print(f"התקבל משתמש: {data['name']}")
        return data

async def fetch_multiple_users():
    """מביא מספר משתמשים במקביל"""
    print("מתחיל לשלוף משתמשים...\n")
    start = time.time()
    
    async with aiohttp.ClientSession() as session:
        # שולפים 5 משתמשים במקביל
        tasks = [fetch_user(session, user_id) for user_id in range(1, 6)]
        users = await asyncio.gather(*tasks)
    
    elapsed = time.time() - start
    print(f"\nשלפנו {len(users)} משתמשים ב-{elapsed:.1f} שניות")
    
    # מדפיס את כל השמות
    print("\nשמות המשתמשים:")
    for user in users:
        print(f"  - {user['name']}")

# הרצה
# asyncio.run(fetch_multiple_users())

<div dir="rtl" style="text-align: right;">

**מה קורה כאן?**
1. שולחים 5 בקשות HTTP **ביחד**
2. לא ממתינים לתשובה של כל אחת
3. כשכל הבקשות חוזרות - מעבדים את התוצאות
4. **זמן:** פי 5 יותר מהיר מאשר אחת אחרי השנייה!

</div>

<div dir="rtl" style="text-align: right;">

## טיפול בשגיאות

מה קורה אם אחת המשימות נכשלת?

</div>

In [ ]:
async def task_with_error(task_num):
    """משימה שעלולה להיכשל"""
    await asyncio.sleep(1)
    
    if task_num == 3:
        raise ValueError(f"שגיאה במשימה {task_num}!")
    
    print(f"משימה {task_num} הצליחה")
    return f"תוצאה {task_num}"

async def run_with_error_handling():
    """מריץ משימות עם טיפול בשגיאות"""
    try:
        tasks = [task_with_error(i) for i in range(1, 6)]
        results = await asyncio.gather(*tasks)
        print(f"כל המשימות הצליחו: {results}")
    except ValueError as e:
        print(f"שגיאה בוצעה: {e}")

asyncio.run(run_with_error_handling())

<div dir="rtl" style="text-align: right;">

**שימו לב:**
- אם אחת המשימות נכשלת, `gather()` יזרוק את השגיאה
- כדי להמשיך גם אם יש שגיאות, אפשר להשתמש ב-`return_exceptions=True`:

</div>

In [ ]:
async def run_with_continue_on_error():
    """ממשיך גם אם יש שגיאות"""
    tasks = [task_with_error(i) for i in range(1, 6)]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    # בודק כל תוצאה
    for i, result in enumerate(results, 1):
        if isinstance(result, Exception):
            print(f"משימה {i} נכשלה: {result}")
        else:
            print(f"משימה {i} הצליחה: {result}")

asyncio.run(run_with_continue_on_error())

<div dir="rtl" style="text-align: right;">

## סיכום

### מה למדנו?

✅ **תכנות אסינכרוני** - הרצת מספר פעולות במקביל  
✅ **async/await** - תחביר לפונקציות אסינכרוניות  
✅ **asyncio.gather()** - הרצת משימות במקביל  
✅ **aiohttp** - בקשות HTTP אסינכרוניות  
✅ **טיפול בשגיאות** - איך לטפל כשמשהו נכשל  

### מתי להשתמש?

**כדאי:**
- הורדת קבצים רבים
- שליחת בקשות ל-APIs
- קריאה/כתיבה של קבצים גדולים
- כל דבר שכולל המתנה

**לא כדאי:**
- חישובים כבדים (CPU)
- משימות פשוטות שלא כוללות המתנה
- כשהקוד נהיה מסובך מדי

### ההבדל העיקרי מ-Threading:

| **Threading** | **asyncio** |
|--------------|-------------|
| Threads אמיתיים | Coroutines (קל יותר) |
| יותר זיכרון | פחות זיכרון |
| מתאים ל-CPU | מתאים ל-I/O |
| יכול להריץ אלפים | יכול להריץ מיליונים! |

---

**בהצלחה! 🚀**

</div>